# 01 — Bronze Ingestion (Free Edition serverless)

**Owner:** Lead • **Phase:** 2 • **Day:** 1 evening – 2 morning

Ingest all 61 SQLite tables into `NEXAMART_DW.NEXAMART_BRONZE` with **zero transformation**.
Add only 3 metadata columns to every table:
- `_SOURCE_TABLE` — string, the source table name
- `_INGESTION_TIMESTAMP` — UTC timestamp of this run
- `_SOURCE_ROW_NUMBER` — 1-based row order from source

**Idempotent** by `overwrite=True`. Re-running produces identical tables.

## Adapted from the brief
Brief Section 7.4 specified Spark-Snowflake Maven JAR + cluster Libraries tab. Free Edition serverless has neither. We use `snowflake-connector-python` + `write_pandas()` instead — same idempotent contract, equivalent semantics, ~3-4 min total runtime for 843,304 rows. See `docs/databricks_setup.md` for the full deviation list.

## Cell 1 — Install the Snowflake Python connector

In [ ]:
%pip install -q snowflake-connector-python pandas
# dbutils.library.restartPython()  # SKIP on Free Edition — kernel restart hangs unreliably

## Cell 2 — Widgets (paste your Snowflake password into `sf_password`)

Free Edition's secrets API is restricted; widgets keep credentials out of Git source.

In [ ]:
dbutils.widgets.text('sf_account',   'rhxendw-yb24678')
dbutils.widgets.text('sf_user',      'NEXAMART_LEAD')
dbutils.widgets.text('sf_password',  '')
dbutils.widgets.text('sf_warehouse', 'NEXAMART_WH')
dbutils.widgets.text('sf_role',      'ACCOUNTADMIN')

## Cell 3 — Configuration

In [ ]:
import sqlite3, pandas as pd
from datetime import datetime
from snowflake.connector import connect
from snowflake.connector.pandas_tools import write_pandas

SQLITE_PATH = '/Volumes/workspace/default/nexamart/nexamart_operations.db'
RUN_TS = datetime.utcnow()
print(f'Run timestamp (UTC): {RUN_TS.isoformat()}')
print(f'SQLite source: {SQLITE_PATH}')

def get_sf():
    return connect(
        account   = dbutils.widgets.get('sf_account'),
        user      = dbutils.widgets.get('sf_user'),
        password  = dbutils.widgets.get('sf_password'),
        warehouse = dbutils.widgets.get('sf_warehouse'),
        role      = dbutils.widgets.get('sf_role'),
        database  = 'NEXAMART_DW',
        schema    = 'NEXAMART_BRONZE',
    )

## Cell 4 — Discover source tables

Should list 61 tables. If anything other than 61, **stop** and reconcile.

In [ ]:
# Unity Catalog Volumes don't support SQLite file locking; copy to /tmp first.
import shutil
LOCAL_PATH = '/tmp/nexamart_operations.db'
shutil.copy(SQLITE_PATH, LOCAL_PATH)
print(f'Copied DB to {LOCAL_PATH} (sqlite3 needs POSIX-locking filesystem)')
con = sqlite3.connect(LOCAL_PATH)
cursor = con.cursor()
tables = [
    row[0]
    for row in cursor.execute(
        "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name"
    ).fetchall()
]
print(f'Discovered {len(tables)} tables in source.')
assert len(tables) == 61, f'Expected 61 tables, got {len(tables)}'

## Cell 5 — Ingestion loop

For each table:
1. Read with sqlite3 + pandas
2. Add 3 metadata columns
3. Uppercase column names (Snowflake default identifier behaviour)
4. `write_pandas(..., overwrite=True, auto_create_table=True)` to NEXAMART_BRONZE

Empty tables (`pc_price_history`) get a CREATE OR REPLACE TABLE with inferred VARCHAR columns — no rows but the schema lands.

In [ ]:
ingestion_results = []

with get_sf() as ctx:
    for t in tables:
        try:
            pdf = pd.read_sql(f'SELECT * FROM "{t}"', con)

            # Coerce object columns to STRING. write_pandas with auto_create_table=True
            # lets snowflake-connector infer column types from pandas dtypes, and a column
            # with mostly numeric values plus the occasional alphanumeric (e.g.
            # si_inventory_movements.OPERATOR_ID has mostly numeric IDs but some 'WH-OP-959'
            # strings) gets inferred as INT64 and the write fails on the first string row.
            # Forcing object→str up front keeps Bronze's "raw mirror" contract: nothing is
            # transformed, but storage type stays compatible with what the source actually has.
            for c in pdf.select_dtypes(include='object').columns:
                pdf[c] = pdf[c].astype(str)

            source_count = len(pdf)

            # 3 metadata columns
            pdf['_SOURCE_TABLE']        = t
            pdf['_INGESTION_TIMESTAMP'] = RUN_TS
            pdf['_SOURCE_ROW_NUMBER']   = list(range(1, source_count + 1))

            # Snowflake uppercase identifiers
            pdf.columns = [c.upper() for c in pdf.columns]

            if source_count == 0:
                # write_pandas refuses empty frames; emulate via DDL
                cols_ddl = ', '.join(f'"{c}" STRING' for c in pdf.columns)
                ctx.cursor().execute(
                    f'CREATE OR REPLACE TABLE NEXAMART_BRONZE.{t.upper()} ({cols_ddl})'
                )
                status = 'OK_EMPTY'
            else:
                success, _, _, _ = write_pandas(
                    ctx, pdf,
                    table_name=t.upper(),
                    schema='NEXAMART_BRONZE',
                    database='NEXAMART_DW',
                    auto_create_table=True,
                    overwrite=True,
                    quote_identifiers=False,
                )
                status = 'OK' if success else 'FAIL'

            ingestion_results.append({'source_table': t, 'row_count': source_count, 'status': status, 'error': None})
            print(f'  {t:35s} {source_count:>7}  {status}')
        except Exception as e:
            ingestion_results.append({'source_table': t, 'row_count': -1, 'status': 'FAIL', 'error': str(e)[:300]})
            print(f'  {t:35s}    FAIL  {e}')

con.close()

ok    = sum(1 for r in ingestion_results if r['status'].startswith('OK'))
fail  = sum(1 for r in ingestion_results if r['status'] == 'FAIL')
total = sum(r['row_count'] for r in ingestion_results if r['row_count'] >= 0)
print(f'\n=== Summary: {ok} OK / {fail} FAIL / {total:,} total rows ingested ===')

## Cell 6 — Write `_INGESTION_LOG`

Tracks per-table outcomes. Read by `sql/bronze_validation.sql`.

In [ ]:
log_pdf = pd.DataFrame(ingestion_results)
log_pdf['ingestion_timestamp'] = RUN_TS
log_pdf.columns = [c.upper() for c in log_pdf.columns]

with get_sf() as ctx:
    success, _, n, _ = write_pandas(
        ctx, log_pdf,
        table_name='_INGESTION_LOG',
        schema='NEXAMART_BRONZE',
        database='NEXAMART_DW',
        auto_create_table=True,
        overwrite=True,
        quote_identifiers=False,
    )
    print(f'_INGESTION_LOG written: {n} rows.')
log_pdf

## Cell 7 — Embedded validation against verified ground-truth counts

Pass = empty `mismatches` list. Total = 843,304.

In [ ]:
EXPECTED = {
    'cl_customers': 2501, 'cl_loyalty_tiers': 4, 'cl_loyalty_transactions': 2860,
    'cs_agents': 25, 'cs_case_events': 225, 'cs_cases': 129, 'cs_complaint_categories': 12,
    'dc_carriers': 5, 'dc_delivery_events': 3542, 'dc_event_types': 10, 'dc_shipments': 771,
    'ec_delivery_methods': 5, 'ec_order_lines': 1840, 'ec_order_status_codes': 9,
    'ec_order_status_history': 3307, 'ec_orders': 963,
    'nl_categories': 13, 'nl_event_types': 14, 'nl_listing_events': 38706,
    'nl_listings': 1253, 'nl_user_accounts': 356,
    'pc_brands': 30, 'pc_categories': 27, 'pc_condition_codes': 10,
    'pc_price_history': 0, 'pc_products': 65,
    'pg_instrument_types': 12, 'pg_status_codes': 8, 'pg_transactions': 963,
    'pos_cashiers': 160, 'pos_payment_methods': 7, 'pos_status_codes': 6,
    'pos_stores': 20, 'pos_transaction_lines': 24507, 'pos_transactions': 10868,
    'rr_refund_events': 74, 'rr_return_reasons': 12, 'rr_return_receipts': 74,
    'rr_return_requests': 74, 'rv_reviews': 377,
    'si_inventory_movements': 438018, 'si_inventory_snapshots': 216645,
    'si_movement_types': 10,
    'ts_fulfilment_events': 593, 'ts_marketplace_orders': 252,
    'ts_report_reasons': 11, 'ts_risk_signals': 58, 'ts_safety_reports': 91,
    'ts_seller_listings': 400, 'ts_seller_status_codes': 5, 'ts_seller_types': 4,
    'ts_sellers': 100, 'ts_signal_types': 10,
    'wh_inbound_receipts': 57, 'wh_inventory_movements': 30437,
    'wh_inventory_snapshots': 38610, 'wh_movement_types': 12, 'wh_warehouses': 3,
    'ws_event_types': 17, 'ws_page_events': 20757, 'ws_sessions': 3370,
}

actual = {r['source_table']: r['row_count'] for r in ingestion_results}
mismatches = [
    {'table': t, 'expected': e, 'actual': actual.get(t), 'diff': (actual.get(t) or 0) - e}
    for t, e in EXPECTED.items() if actual.get(t) != e
]

expected_total = sum(EXPECTED.values())
actual_total   = sum(v for v in actual.values() if v >= 0)
print(f'Expected total: {expected_total:,}')
print(f'Actual total:   {actual_total:,}')
print(f'Tables expected: {len(EXPECTED)}; loaded: {len(actual)}')

if mismatches:
    print('\n=== ❌ MISMATCHES ===')
    for m in mismatches:
        print(f'  {m["table"]}: expected {m["expected"]:>7}, got {m["actual"]} (diff {m["diff"]:+})')
    raise AssertionError(f'{len(mismatches)} table(s) failed Bronze validation')
else:
    print('\n=== ✅ Bronze validation PASSED — all 61 tables match expected row counts ===')

## Next steps

1. Run `sql/bronze_validation.sql` in Snowflake console — empty result on the assertion query.
2. Tag the Git commit `bronze-locked`.
3. Proceed to Phase 3: shared Silver utilities + `seed_status_mapping.ipynb`.
4. Notify members in standup that they can begin Silver against `NEXAMART_BRONZE`.